# V1-01: OpenDART 데이터 기준선

이 노트북은 V1의 첫 실제 프로젝트 확인 단계입니다. 아직 LLM, LangChain, LangGraph를 사용하지 않습니다.

확인할 흐름은 다음과 같습니다.

`기업명 → corp_code → 사업보고서 재무제표 → 연결/별도 기준 → 결정적 계산 → 기간 공시 목록 → 사업보고서 원문 링크`

여기서의 `assert`는 성능 평가가 아니라, V1 결과에 필요한 최소 필드가 실제 응답에 있는지 확인하는 실행 점검입니다.

## 0. V1 데이터 기준

- 사업연도 재무 수치는 사업보고서(`11011`) 기준으로 조회합니다.
- 연결재무제표(`CFS`)를 먼저 조회하고, 값이 없을 때만 별도재무제표(`OFS`)로 대체합니다.
- 공시 근거는 사업보고서 원문 링크를 기본으로 합니다. 기간 공시 목록은 사용자가 요청했을 때 함께 보여줄 수 있는 원천 데이터입니다.

API의 필드와 원천 제한은 [V1 사전 레퍼런스](../../docs/v1/REFERENCES.md)를 먼저 확인하세요.

## 1. 표준 라이브러리와 기존 OpenDART client 불러오기

재무 조회와 공시 목록 조회는 검증된 `src/dart/client.py`를 재사용합니다. notebook에서는 반환된 raw 행의 필드와 V1 데이터 계약만 확인합니다.

In [1]:
import os
import sys
from pathlib import Path

## 2. 프로젝트 루트와 `.env` 확인

이 notebook은 프로젝트 루트, `notebooks/`, `notebooks/v1/` 중 어디에서 열어도 프로젝트 루트를 찾습니다. API 키 값은 출력하지 않습니다.

In [2]:
current_dir = Path.cwd().resolve()
if current_dir.name == "v1":
    project_root = current_dir.parents[1]
elif current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    project_root = current_dir

env_path = project_root / ".env"
print("프로젝트 루트:", project_root)
print(".env 존재 여부:", env_path.exists())

프로젝트 루트: /home/sms/openclaw_file/FinSight
.env 존재 여부: True


In [3]:
if not env_path.exists():
    raise FileNotFoundError(f"{env_path} 파일을 먼저 만들어 주세요.")

for raw_line in env_path.read_text(encoding="utf-8").splitlines():
    line = raw_line.strip()
    if not line or line.startswith("#") or "=" not in line:
        continue
    key, value = line.split("=", 1)
    os.environ.setdefault(key.strip(), value.strip().strip("'\""))

api_key = os.environ.get("OPENDART_API_KEY")
if not api_key:
    raise ValueError(".env에 OPENDART_API_KEY를 설정해 주세요.")

print("OPENDART_API_KEY를 읽었습니다. 값은 출력하지 않습니다.")

OPENDART_API_KEY를 읽었습니다. 값은 출력하지 않습니다.


In [4]:
sys.path.insert(0, str(project_root / "src"))

from dart.client import (
    build_filing_url,
    fetch_disclosure_rows,
    fetch_financial_rows,
    find_corp_info,
    summarize_financial_rows,
)

## 3. 조회 조건 정하기

`business_years`는 재무제표의 사업연도이고, `disclosure_start`·`disclosure_end`는 공시가 실제 접수된 날짜입니다. 두 날짜 기준은 다를 수 있습니다.

In [5]:
corp_name = "삼성전자"
business_years = [2023, 2024, 2025]
report_code = "11011"  # 사업보고서
disclosure_start = "20240101"
disclosure_end = "20260331"

## 4. 기업명에서 `corp_code` 찾기

V1의 입력인 기업명은 OpenDART 재무·공시 API에 바로 전달되지 않습니다. 먼저 고유번호로 변환해야 합니다.

In [6]:
corp_info = find_corp_info(corp_name, api_key)
corp_code = corp_info["corp_code"]

{
    "기업명": corp_info["corp_name"],
    "corp_code": corp_code,
    "종목코드": corp_info.get("stock_code"),
}

{'기업명': '삼성전자', 'corp_code': '00126380', '종목코드': '005930'}

## 5. 연결 우선 재무제표 조회 정책

이 함수는 V1의 선택 규칙을 눈으로 확인하기 위한 notebook 코드입니다. `CFS`가 없다는 OpenDART 응답(`013`)이 오거나 행 목록이 비어 있을 때만 `OFS`를 다시 조회합니다. 네트워크·인증 오류는 숨기지 않고 그대로 중단합니다.

반복 실행으로 정책이 검증되면, 이 부분만 `client.py`에 옮길지 결정합니다.

In [7]:
def fetch_with_statement_policy(corp_code, business_year, api_key):
    try:
        cfs_rows = fetch_financial_rows(
            corp_code, business_year, api_key, report_code=report_code, fs_div="CFS"
        )
    except RuntimeError as error:
        if "[013]" not in str(error):
            raise
        cfs_rows = []

    if cfs_rows:
        return "CFS", cfs_rows

    ofs_rows = fetch_financial_rows(
        corp_code, business_year, api_key, report_code=report_code, fs_div="OFS"
    )
    return "OFS", ofs_rows

## 6. 첫 사업연도의 raw 계정 행 확인

raw 행에서 특히 `account_id`, `account_nm`, `thstrm_amount`, `currency`, `rcept_no`, `fs_div`를 확인하세요. `rcept_no`는 해당 재무제표 근거를 추적할 때 사용합니다.

In [8]:
first_basis, first_rows = fetch_with_statement_policy(
    corp_code, business_years[0], api_key
)

print(f"재무제표 기준: {first_basis}")
first_rows[0]

재무제표 기준: CFS


{'rcept_no': '20240312000736',
 'reprt_code': '11011',
 'bsns_year': '2023',
 'corp_code': '00126380',
 'sj_div': 'BS',
 'sj_nm': '재무상태표',
 'account_id': 'ifrs-full_Assets',
 'account_nm': '자산총계',
 'account_detail': '-',
 'thstrm_nm': '제 55 기',
 'thstrm_amount': '455905980000000',
 'frmtrm_nm': '제 54 기',
 'frmtrm_amount': '448424507000000',
 'bfefrmtrm_nm': '제 53 기',
 'bfefrmtrm_amount': '426621158000000',
 'ord': '7',
 'currency': 'KRW'}

## 7. 여러 사업연도의 주요 재무항목 정리

`client.py`의 요약 함수가 매출액·영업이익·당기순이익을 같은 형식으로 정리합니다. 여기서는 각 연도의 실제 재무제표 기준과 접수번호를 함께 남깁니다.

In [9]:
financial_summaries = []

for business_year in business_years:
    statement_basis, rows = fetch_with_statement_policy(
        corp_code, business_year, api_key
    )
    summary = summarize_financial_rows(corp_info["corp_name"], business_year, rows)
    summary["보고서 코드"] = report_code
    summary["재무제표 기준"] = statement_basis
    summary["접수번호"] = rows[0].get("rcept_no")
    financial_summaries.append(summary)

financial_summaries

[{'기업명': '삼성전자',
  '사업연도': 2023,
  '매출액': '258935494000000',
  '영업이익': '6566976000000',
  '당기순이익': '15487100000000',
  '보고서 코드': '11011',
  '재무제표 기준': 'CFS',
  '접수번호': '20240312000736'},
 {'기업명': '삼성전자',
  '사업연도': 2024,
  '매출액': '300870903000000',
  '영업이익': '32725961000000',
  '당기순이익': '34451351000000',
  '보고서 코드': '11011',
  '재무제표 기준': 'CFS',
  '접수번호': '20250311001085'},
 {'기업명': '삼성전자',
  '사업연도': 2025,
  '매출액': '333605938000000',
  '영업이익': '43601051000000',
  '당기순이익': '45206805000000',
  '보고서 코드': '11011',
  '재무제표 기준': 'CFS',
  '접수번호': '20260310002820'}]

## 7-1. 계산 전 최소 데이터 계약 점검

계산 전에 필수 수치가 모두 있는지 먼저 확인합니다. 누락이 있으면 숫자 변환이나 나눗셈 오류 대신, 계약이 충족되지 않았다는 지점에서 중단됩니다.

In [10]:
assert len(financial_summaries) == len(business_years)
assert all(row["보고서 코드"] == report_code for row in financial_summaries)
assert all(row["재무제표 기준"] in {"CFS", "OFS"} for row in financial_summaries)
assert all(row["접수번호"] for row in financial_summaries)
assert all(row[label] is not None for row in financial_summaries for label in ["매출액", "영업이익", "당기순이익"])

print("재무 데이터 계약 점검을 통과했습니다.")

재무 데이터 계약 점검을 통과했습니다.


## 8. Python으로 결정적 계산하기

계산은 LLM에게 맡기지 않습니다. OpenDART 원문 문자열을 정수로 바꾼 뒤, 매출 성장률과 영업이익률을 Python으로 계산합니다.

In [11]:
def amount_to_int(value):
    return int(value.replace(",", ""))

trend_rows = []
previous_revenue = None

for summary in financial_summaries:
    revenue = amount_to_int(summary["매출액"])
    operating_income = amount_to_int(summary["영업이익"])
    revenue_growth = None if previous_revenue is None else (revenue / previous_revenue - 1) * 100

    trend_rows.append(
        {
            "사업연도": summary["사업연도"],
            "매출 성장률(%)": None if revenue_growth is None else round(revenue_growth, 2),
            "영업이익률(%)": round(operating_income / revenue * 100, 2),
        }
    )
    previous_revenue = revenue

trend_rows

[{'사업연도': 2023, '매출 성장률(%)': None, '영업이익률(%)': 2.54},
 {'사업연도': 2024, '매출 성장률(%)': 16.2, '영업이익률(%)': 10.88},
 {'사업연도': 2025, '매출 성장률(%)': 10.88, '영업이익률(%)': 13.07}]

## 9. 기간 공시 목록 raw 응답 확인

공시 목록은 `client.py`에서 모든 페이지를 조회합니다. 여기서는 반환 행의 `report_nm`, `rcept_no`, `rcept_dt` 형태를 확인합니다. API 키가 포함된 URL은 출력하지 않습니다.

In [12]:
# 공시 목록의 HTTP 요청·페이지 처리 로직은 client.py에 있습니다.

In [ ]:
disclosure_rows = fetch_disclosure_rows(
    corp_code,
    disclosure_start,
    disclosure_end,
    api_key,
    final_reports_only=True,
)

print(f"공시 수: {len(disclosure_rows)}")
disclosure_rows[:3]

## 9-1. 모든 공시 페이지 가져오기

OpenDART 공시검색은 한 페이지에 최대 100건을 반환합니다. `fetch_disclosure_rows()`는 `total_page`까지 조회하고 `total_count`와 실제 행 수가 같은지 확인합니다. 지금은 최종 제출본(`last_reprt_at=Y`)만 조회하므로 정정 전 문서는 포함하지 않습니다.

In [14]:
print(f"전체 공시 수: {len(disclosure_rows)}")

전체 공시 수: 2908


## 10. 사업보고서 원문 링크 만들기

V1은 공시의 중요도를 자동으로 추정하지 않습니다. 여기서는 사업보고서 제목이 포함된 공시를 찾아, 사용자가 직접 검토할 원문 링크를 만듭니다.

In [15]:
business_report_evidence = [
    {
        "공시명": row["report_nm"],
        "접수일": row["rcept_dt"],
        "접수번호": row["rcept_no"],
        "원문 링크": build_filing_url(row["rcept_no"]),
    }
    for row in disclosure_rows
    if "사업보고서" in row.get("report_nm", "")
]

business_report_evidence

[{'공시명': '사업보고서 (2025.12)',
  '접수일': '20260310',
  '접수번호': '20260310002820',
  '원문 링크': 'https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20260310002820'},
 {'공시명': '사업보고서 (2024.12)',
  '접수일': '20250311',
  '접수번호': '20250311001085',
  '원문 링크': 'https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20250311001085'},
 {'공시명': '사업보고서 (2023.12)',
  '접수일': '20240312',
  '접수번호': '20240312000736',
  '원문 링크': 'https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20240312000736'}]

## 11. 최소 데이터 계약 점검

아래는 성능 평가지표가 아닙니다. V1의 결과가 기업·연도·재무제표 기준·핵심 수치·공시 식별자를 실제로 확보했는지 확인하는 최소 점검입니다.

In [16]:
assert business_report_evidence
assert all(row["원문 링크"].startswith("https://dart.fss.or.kr/") for row in business_report_evidence)

print("V1 최소 데이터 계약 점검을 통과했습니다.")

V1 최소 데이터 계약 점검을 통과했습니다.


## 이번 notebook에서 확인할 것

1. 기업명은 `corp_code`로 변환된 뒤 재무·공시 API에 전달됩니다.
2. 같은 연도라도 연결(`CFS`)과 별도(`OFS`) 기준은 다를 수 있으므로 결과에 남겨야 합니다.
3. 재무제표의 금액은 문자열이므로, 계산 전에 Python 숫자로 바꿉니다.
4. 사업연도와 공시 접수일은 서로 다를 수 있습니다.
5. `rcept_no`는 재무·공시 결과를 원문으로 되돌아갈 수 있게 하는 핵심 식별자입니다.

다음 단계에서는 여러 기업·연도의 정답 근거를 고정한 뒤 재무 조회와 계산의 정확도를 측정합니다. 아직 Router, RAG, LangGraph workflow는 추가하지 않습니다.